# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barak-hub/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice: Decision Tree Classifier

My chosen method is a Decision Tree Classifier for my Google Search Ranking & Discoverability lane.

The goal is to classify webpages as likely to experience declining search impressions (1) or non-declining impressions (0).

I chose a Decision Tree because it can learn patterns from historical webpage features and its decisions are relatively easy to explain to SEO specialists.

I will evaluate the model using F1-score, precision, and recall on held-out data. I will use only information available before the prediction period to avoid data leakage.

The model will be compared against a simple baseline. Its usefulness will depend on whether it improves prediction performance on unseen data.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design: Client-Grouped Train/Test Split

I will use a client-grouped train/test split, allocating approximately 80% of clients to training and 20% to testing.

The dataset contains 52 unique clients and 133,852 unique webpages across 2,414,248 records.

I chose a grouped split because multiple webpages and search queries belong to the same client. Keeping each client entirely within one split reduces the risk of client-specific information leaking between training and testing.

The dataset contains only one 90-day observation window, from April 2 to June 30, 2026. Therefore, a chronological split across multiple observation windows is not possible.

This evaluation will measure how well the model generalizes to unseen clients, rather than how accurately it predicts future changes in search impressions.

I will use a fixed random seed for reproducibility and evaluate the model using F1-score, precision, and recall.

Before training, I will verify that the target labels and input features are appropriate and do not introduce data leakage.

In [2]:

import pandas as pd
from google.colab import userdata
from huggingface_hub import hf_hub_download

token = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=token
)

df = pd.read_parquet(file_path)

print("Dataset columns:")
print(df.columns.tolist())

print("\nDataset shape:")
print(df.shape)

print("\nFirst five records:")
display(df.head())


Dataset columns:
['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']

Dataset shape:
(2414248, 21)

First five records:


,client_hash_id,content_hash_id,query_hash_id,query_char_count,query_token_count,window_start,window_end,impressions_90d,clicks_90d,impressions_last30,...,impressions_prev30,clicks_prev30,avg_position_90d,avg_position_last30,avg_position_prev30,content_total_impressions_90d,content_visible_query_count,rare_query_count,rare_impressions_share,anonymized_impressions_share
0,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_58b1b001f839d699,17,3,2026-04-02,2026-06-30,11,0,0,...,11,0,10.818182,NaN,10.818182,1466,14,32,0.043656,0.725102
1,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_922b8eca2a24cd34,34,7,2026-04-02,2026-06-30,13,0,0,...,1,0,1.769231,NaN,11.000000,1466,14,32,0.043656,0.725102
2,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_9f0c36a6ae2a6a99,16,2,2026-04-02,2026-06-30,16,0,11,...,5,0,23.562500,24.272727,22.000000,1466,14,32,0.043656,0.725102
3,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_a032820b5467e996,24,4,2026-04-02,2026-06-30,55,0,1,...,1,0,2.200000,13.000000,0.000000,1466,14,32,0.043656,0.725102
4,client_08a6a72ff48e62c0,content_447894f2faf0d2bc,query_ba1a2f131961c5da,18,3,2026-04-02,2026-06-30,14,0,0,...,0,0,3.428571,NaN,NaN,1466,14,32,0.043656,0.725102


In [3]:

print("Unique clients:", df["client_hash_id"].nunique())

print("Unique webpages:", df["content_hash_id"].nunique())

print("Unique observation windows:")
print(df[["window_start", "window_end"]].drop_duplicates())

print("\nMissing values in key columns:")
print(df[
    ["client_hash_id", "content_hash_id",
     "window_start", "window_end",
     "impressions_prev30", "impressions_last30"]
].isnull().sum())


Unique clients: 52
Unique webpages: 133852
Unique observation windows:
  window_start  window_end
0   2026-04-02  2026-06-30

Missing values in key columns:
client_hash_id        0
content_hash_id       0
window_start          0
window_end            0
impressions_prev30    0
impressions_last30    0
dtype: int64


In [4]:

from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=df["client_hash_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

print("Training records:", len(train_df))
print("Testing records:", len(test_df))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Clients in both groups:", len(train_clients & test_clients))

assert train_clients.isdisjoint(test_clients)
print("Client-grouped split verified!")


Training records: 2164186
Testing records: 250062
Training clients: 41
Testing clients: 11
Clients in both groups: 0
Client-grouped split verified!


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [6]:

# Exploratory label: observed decline within the 90-day window
df["observed_decline"] = (
    df["impressions_last30"] < df["impressions_prev30"]
).astype(int)

print("Observed decline distribution:")
print(df["observed_decline"].value_counts())

print("\nObserved decline percentages:")
print(
    df["observed_decline"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nMissing labels:")
print(df["observed_decline"].isna().sum())


Observed decline distribution:
observed_decline
1    1373407
0    1040841
Name: count, dtype: int64

Observed decline percentages:
observed_decline
1    56.89
0    43.11
Name: proportion, dtype: float64

Missing labels:
0


In [7]:

page_groups = df.groupby(
    ["client_hash_id", "content_hash_id"]
)

print("Unique client-webpage combinations:",
      page_groups.ngroups)

label_counts = page_groups["observed_decline"].nunique()

print("Webpages with consistent query labels:",
      (label_counts == 1).sum())

print("Webpages with mixed query labels:",
      (label_counts > 1).sum())

print("Percentage with mixed labels:",
      round((label_counts > 1).mean() * 100, 2))


Unique client-webpage combinations: 133852
Webpages with consistent query labels: 51279
Webpages with mixed query labels: 82573
Percentage with mixed labels: 61.69


In [8]:

# Aggregate search-query records into webpages
page_df = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions_prev30=("impressions_prev30", "sum"),
        impressions_last30=("impressions_last30", "sum"),
        clicks_prev30=("clicks_prev30", "sum"),
        query_count=("query_hash_id", "nunique")
    )
)

# Create one observed-decline label per webpage
page_df["declining_impressions"] = (
    page_df["impressions_last30"]
    < page_df["impressions_prev30"]
).astype(int)

print("Total webpages:", len(page_df))

print("\nWebpage-level label distribution:")
print(page_df["declining_impressions"].value_counts())

print("\nWebpage-level label percentages:")
print(
    page_df["declining_impressions"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(page_df.head())


Total webpages: 133852

Webpage-level label distribution:
declining_impressions
1    80534
0    53318
Name: count, dtype: int64

Webpage-level label percentages:
declining_impressions
1    60.17
0    39.83
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,impressions_prev30,impressions_last30,clicks_prev30,query_count,declining_impressions
0,client_06d356715a8ff3b6,content_0058bd88fb1821f2,0,16,0,1,0
1,client_06d356715a8ff3b6,content_0059a4d4195810c9,139,123,0,12,1
2,client_06d356715a8ff3b6,content_005b6b7f7b8dda7f,59,38,0,9,1
3,client_06d356715a8ff3b6,content_0094c7d0fbcc07b7,0,28,0,2,0
4,client_06d356715a8ff3b6,content_00a34394d4ee05ce,9,6,0,1,1


In [9]:

# Inspect candidate model features
features = [
    "impressions_prev30",
    "clicks_prev30",
    "query_count"
]

print("Feature summary:")
display(page_df[features].describe())

print("\nMissing values:")
print(page_df[features].isna().sum())

print("\nTarget distribution:")
print(page_df["declining_impressions"].value_counts())


Feature summary:


,impressions_prev30,clicks_prev30,query_count
count,133852.000000,133852.000000,133852.000000
mean,562.013537,1.247116,18.036697
std,3132.413744,9.406557,47.037384
min,0.000000,0.000000,1.000000
25%,10.000000,0.000000,2.000000
50%,49.000000,0.000000,6.000000
75%,220.000000,0.000000,17.000000
max,313945.000000,758.000000,7889.000000



Missing values:
impressions_prev30    0
clicks_prev30         0
query_count           0
dtype: int64

Target distribution:
declining_impressions
1    80534
0    53318
Name: count, dtype: int64


In [10]:

# Use the same clients from our verified split
page_train = page_df[
    page_df["client_hash_id"].isin(train_clients)
].copy()

page_test = page_df[
    page_df["client_hash_id"].isin(test_clients)
].copy()

# Select provisional features
feature_cols = [
    "impressions_prev30",
    "clicks_prev30"
]

target_col = "declining_impressions"

X_train = page_train[feature_cols]
X_test = page_test[feature_cols]

y_train = page_train[target_col]
y_test = page_test[target_col]

print("Training webpages:", len(X_train))
print("Testing webpages:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True).round(3))

print("\nClient overlap:",
      len(
          set(page_train["client_hash_id"])
          & set(page_test["client_hash_id"])
      ))

assert len(X_train) + len(X_test) == len(page_df)
assert set(page_train["client_hash_id"]).isdisjoint(
    set(page_test["client_hash_id"])
)


Training webpages: 117944
Testing webpages: 15908

Training target distribution:
declining_impressions
1    0.598
0    0.402
Name: proportion, dtype: float64

Testing target distribution:
declining_impressions
1    0.629
0    0.371
Name: proportion, dtype: float64

Client overlap: 0


In [11]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Create our Decision Tree
model = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=100,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Predict on unseen clients
y_pred = model.predict(X_test)

# Evaluate the model
print("Decision Tree Results")
print("---------------------")

print("Precision:", round(
    precision_score(y_test, y_pred), 4
))

print("Recall:", round(
    recall_score(y_test, y_pred), 4
))

print("F1-score:", round(
    f1_score(y_test, y_pred), 4
))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Decision Tree Results
---------------------
Precision: 0.7262
Recall: 0.9378
F1-score: 0.8185

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.40      0.53      5897
           1       0.73      0.94      0.82     10011

    accuracy                           0.74     15908
   macro avg       0.76      0.67      0.67     15908
weighted avg       0.75      0.74      0.71     15908



In [12]:

import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# Learn the majority class from training data only
majority_class = y_train.mode().iloc[0]

# Predict the same class for every test webpage
baseline_pred = np.full(
    len(y_test),
    majority_class
)

print("Majority class:", majority_class)

print("\nBaseline Results")
print("---------------------")

print("Precision:", round(
    precision_score(y_test, baseline_pred, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_test, baseline_pred, zero_division=0), 4
))

print("F1-score:", round(
    f1_score(y_test, baseline_pred, zero_division=0), 4
))

print("\nDecision Tree F1:",
      round(f1_score(y_test, y_pred), 4))

print("F1 improvement:",
      round(
          f1_score(y_test, y_pred)
          - f1_score(y_test, baseline_pred),
          4
      ))


Majority class: 1

Baseline Results
---------------------
Precision: 0.6293
Recall: 1.0
F1-score: 0.7725

Decision Tree F1: 0.8185
F1 improvement: 0.046


### Model Evaluation and Baseline Comparison

I trained a Decision Tree Classifier using previous-period impressions and clicks to classify observed declines in search impressions.

The model was evaluated on 15,908 webpages belonging to clients excluded from training.

**Decision Tree results:**
- Precision: 0.7262
- Recall: 0.9378
- F1-score: 0.8185

**Majority-class baseline results:**
- Precision: 0.6293
- Recall: 1.0000
- F1-score: 0.7725

The Decision Tree improved F1-score by 0.0460 (4.60 percentage points) compared with the majority-class baseline.

The improvement suggests that the model identifies useful patterns beyond simply predicting the majority class. However, the Decision Tree sacrifices some recall in exchange for higher precision.

**Limitations:**

1. The dataset contains only one 90-day observation window, so this experiment evaluates observed declines rather than validated future predictions.
2. The client-grouped split prevents client overlap but does not provide a chronological validation.
3. The model uses only two historical-period features, and feature availability must be confirmed before deployment.
4. The majority-class baseline is a new comparison baseline. It is not equivalent to the CTR-prioritization baseline developed in Week 4.
5. The experiment does not establish how well the model will perform on future search data.

**Conclusion:**

The Decision Tree outperformed the majority-class baseline on the held-out client test set. Further time-based validation and a final leakage audit are necessary before interpreting the model as a reliable forecasting system.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [13]:

from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["Actual: No decline", "Actual: Decline"],
    columns=["Predicted: No decline", "Predicted: Decline"]
)

print("Decision Tree Confusion Matrix:")
display(cm_df)

print("\nFeature importance:")
for feature, importance in zip(
    feature_cols, model.feature_importances_
):
    print(f"{feature}: {importance:.4f}")


Decision Tree Confusion Matrix:


,Predicted: No decline,Predicted: Decline
Actual: No decline,2357,3540
Actual: Decline,623,9388



Feature importance:
impressions_prev30: 0.9837
clicks_prev30: 0.0163


### Error Analysis and Model Interpretation

I evaluated my Decision Tree Classifier using a confusion matrix and feature importance analysis on 15,908 test webpages.

**Confusion Matrix Results:**
- True Positives: 9,388
- True Negatives: 2,357
- False Positives: 3,540
- False Negatives: 623

The model correctly identified most webpages that experienced declining impressions. However, it incorrectly predicted declines for 3,540 webpages that did not experience a decline.

These false positives could lead SEO teams to investigate webpages unnecessarily. The 623 false negatives represent declining webpages that the model failed to identify, potentially causing important SEO issues to be overlooked.

**Feature Importance:**
- Previous 30-day impressions: 98.37%
- Previous 30-day clicks: 1.63%

The Decision Tree relies heavily on previous search impressions when making classifications. Previous clicks contribute relatively little to the model's decisions.

These feature importance scores describe how the trained tree uses its inputs. They do not establish causal relationships.

**Interpretation and Limitations:**

The model achieved an F1-score of 0.8185, outperforming the majority-class baseline of 0.7725.

However, the model produces a substantial number of false positives, and its performance has only been evaluated using observed impression changes within one historical observation window.

The results should therefore be treated as exploratory evidence rather than proof of future predictive accuracy.

**Conclusion:**

The Decision Tree provides a measurable improvement over a simple majority-class baseline, but further validation, additional historical observation windows, and careful feature-leakage checks are required before the model can support real-world SEO decisions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.